# 02 — Conversation Buffer Memory

Give an LLM a **rolling history of all messages** so it always knows what was said.

### What you will learn
| Concept | Description |
|---------|-------------|
| Message types | `HumanMessage`, `AIMessage`, `SystemMessage` |
| Buffer memory | Keep every message — full context |
| Window memory | Keep only the last N messages — saves tokens |
| Token problem | Why you can't keep every message forever |

### How it works
```
Turn 1:  [System] + [Human: "Hi, I'm Ravi"]  → LLM → [AI: "Hello Ravi!"]
Turn 2:  [System] + [Human: "Hi, I'm Ravi"] + [AI: "Hello Ravi!"]
                 + [Human: "What's my name?"] → LLM → [AI: "Your name is Ravi."]
```
The **entire previous conversation** is passed in every time.

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install & Import

In [ ]:
# !pip install langchain-groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 2. Set Up the LLM

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0.7)

# Quick test — one message, no history
response = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print("LLM response:", response.content)

## 3. The Buffer — A Simple List

The buffer is just a **Python list of message objects**.
Before every LLM call, the full buffer is included in the request.

In [ ]:
# ── Our conversation buffer ───────────────────────────────────────────
buffer = []

SYSTEM_MSG = SystemMessage(content=(
    "You are a friendly and helpful assistant. "
    "Remember everything the user tells you about themselves."
))

def chat(user_input):
    """Send a message, get a reply, and update the buffer."""
    # 1. Add the user's message to the buffer
    buffer.append(HumanMessage(content=user_input))

    # 2. Build the full message list: system + full history
    messages = [SYSTEM_MSG] + buffer

    # 3. Call the LLM with the entire history
    response = llm.invoke(messages)

    # 4. Add the AI reply to the buffer
    buffer.append(AIMessage(content=response.content))

    # 5. Print the conversation
    print(f"You : {user_input}")
    print(f"Bot : {response.content}")
    print(f"[Buffer size: {len(buffer)} messages]\n")
    return response.content

print("Buffer memory chatbot ready!")

## 4. Have a Multi-Turn Conversation

In [ ]:
# The bot should remember earlier messages throughout this conversation

chat("Hi! My name is Sneha.")
chat("I work as a data scientist in Bangalore.")
chat("My favourite programming language is Python.")

In [ ]:
# Now ask questions about what was said earlier
chat("What is my name?")
chat("Where do I work?")
chat("What language do I prefer?")

## 5. Inspect the Buffer

In [ ]:
print(f"Total messages in buffer: {len(buffer)}\n")
print("Full conversation history:")
print("─" * 45)
for i, msg in enumerate(buffer):
    role = "You" if isinstance(msg, HumanMessage) else "Bot"
    preview = msg.content[:70] + "..." if len(msg.content) > 70 else msg.content
    print(f"  [{i+1:2d}] {role:3s} : {preview}")

## 6. The Token Problem

Every message you add makes the next LLM call **more expensive and slower**.
A buffer with 100 messages sends 100 messages to the API on every turn.

**Solution: Window Memory** — only keep the last N messages.

In [ ]:
MAX_WINDOW = 6    # keep only the last 6 messages (3 exchanges)

window_buffer = []

def chat_with_window(user_input):
    """Same as chat(), but trims the buffer to MAX_WINDOW messages."""    global window_buffer

    window_buffer.append(HumanMessage(content=user_input))

    # ── Trim: keep only the most recent MAX_WINDOW messages ──
    if len(window_buffer) > MAX_WINDOW:
        dropped = len(window_buffer) - MAX_WINDOW
        window_buffer = window_buffer[-MAX_WINDOW:]
        print(f"  [Window] Dropped {dropped} old message(s) to stay within window")

    messages  = [SYSTEM_MSG] + window_buffer
    response  = llm.invoke(messages)
    window_buffer.append(AIMessage(content=response.content))

    print(f"You : {user_input}")
    print(f"Bot : {response.content}")
    print(f"[Window size: {len(window_buffer)}/{MAX_WINDOW} messages]\n")
    return response.content

print("Window memory chatbot ready! (window =", MAX_WINDOW, "messages)")

In [ ]:
# Start a conversation with window memory
chat_with_window("Hello! I'm Karan.")
chat_with_window("I love hiking and photography.")
chat_with_window("I'm planning a trip to Manali next month.")
chat_with_window("What did I say about my trip?")       # within window — knows
chat_with_window("What hobbies did I mention?")         # within window — knows

## 7. Buffer vs Window — Side by Side

| Feature | Buffer Memory | Window Memory |
|---------|--------------|---------------|
| What's kept | Every message ever | Only last N messages |
| Tokens used | Grows forever | Fixed cap |
| Remembers old info | ✅ Yes | ❌ No (old info is dropped) |
| Good for | Short conversations | Long conversations |

**Rule of thumb:** use window memory with `N = 6–10` for most chatbots.

**Next notebook →** instead of dropping old messages, *summarise* them.